In [ ]:
!pip install transformers peft trl bitsandbytes datasets accelerate

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 4bit量子化の設定(QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_name = "Ichiyou1922/Gemma-2-Llama-Swallow-9b-abliterated"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="mashiro_conversation_v2.json", split="train")

# SFTTrainerが認識する "messages" 形式に変換するだけでOK
def to_messages(example):
    return {
        "messages": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

dataset = dataset.map(to_messages)

# 確認
print(dataset[0]["messages"])


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        max_seq_length=2048,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit",
        seed=3407,
    ),
)



In [ ]:
# 学習実行
trainer_stats = trainer.train()

In [ ]:
end_of_turn_token_id = tokenizer.encode("<end_of_turn>", add_special_tokens=False)[0]
start_of_turn_token_id = tokenizer.encode("<start_of_turn>", add_special_tokens=False)[0]

print(f"End of turn token ID: {end_of_turn_token_id}")
print(f"Start of turn token ID: {start_of_turn_token_id}")

In [ ]:
# テスト推論
test_prompts = [
    "今日の配信どうだった？",
    "僕のこと好き？",
    "眠いよ",
    "世界征服の計画は？",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt =True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    # generateには **inputs で展開して渡す
    # do_sample=True, temperature, top_p, repetition_penalty を追加して自然な生成にする
    outputs = model.generate(
        **inputs, 
        max_new_tokens=128, 
        use_cache=True,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    # 入力部分を除外してデコード (inputsは辞書なので .input_ids.shape を見る)
    input_len = inputs.input_ids.shape[-1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    print(f"User: {prompt}")
    print(f"ましろ: {response}\n")


In [ ]:
# 1. LoRAアダプターだけ保存
model.save_pretrained("mashiro_lora")
tokenizer.save_pretrained("mashiro_lora")

# 2. メモリ解放
del model
torch.cuda.empty_cache()

# 3. bf16でベースモデル+LoRAを再ロードしてマージ
from peft import AutoPeftModelForCausalLM

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    "mashiro_lora",               # ← ここにアダプターのパスを渡す
    torch_dtype=torch.bfloat16,   # ← 4bitではなくbf16で読み込む
    device_map="auto",
)
merged_model = merged_model.merge_and_unload()

# 4. ローカルに保存（GGUF変換用）
merged_model.save_pretrained("mashiro_merged")
tokenizer.save_pretrained("mashiro_merged")

# 5. HuggingFace Hubにもpush
merged_model.push_to_hub("Ichiyou1922/mashiro-ai-v7", private=True)
tokenizer.push_to_hub("Ichiyou1922/mashiro-ai-v7", private=True)

print("マージ＋アップロード完了")


In [ ]:
# llama.cppを使ってGGUF変換
!git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt

# HuggingFace形式 → GGUF (bf16)
!python llama.cpp/convert_hf_to_gguf.py mashiro_merged \
    --outtype bf16 \
    --outfile mashiro_bf16.gguf


In [ ]:
# llama.cppのquantizeツールをビルド
!cd llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j$(nproc)

# bf16 → Q4_K_M に量子化
!./llama.cpp/build/bin/llama-quantize mashiro_bf16.gguf mashiro_q4_k_m.gguf Q4_K_M

# 確認
!ls -lh mashiro_q4_k_m.gguf


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="mashiro_q4_k_m.gguf",
    path_in_repo="mashiro_q4_k_m.gguf",
    repo_id="Ichiyou1922/mashiro-ai-v7",
)
print("GGUFアップロード完了")
